# Hansen Ch.18 Difference in Differences

理论见同目录 md（**18.1–18.8**）。本 notebook：CK1994 / DS2004 / BMN2016 实证。

In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from numpy.linalg import inv, pinv
from scipy import stats

ROOT = Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data")


def cluster_ols(y, X, clusters, add_const=False):
    y = np.asarray(y, float)
    X = np.asarray(X, float)
    if add_const:
        X = np.column_stack([np.ones(len(y)), X])
    b = pinv(X.T @ X) @ (X.T @ y)
    e = y - X @ b
    meat = np.zeros((X.shape[1], X.shape[1]))
    for g in pd.unique(clusters):
        m = clusters == g
        score = X[m].T @ e[m]
        meat += np.outer(score, score)
    n, k = X.shape
    nG = len(pd.unique(clusters))
    bread = pinv(X.T @ X)
    V = (nG / (nG - 1)) * ((n - 1) / (n - k)) * bread @ meat @ bread
    se = np.sqrt(np.maximum(np.diag(V), 0))
    return b, se, V


def fe_reg(df, ycol, xcols, fe_col, cluster_col):
    d = df.copy()
    cols = [ycol] + list(xcols)
    means = d.groupby(fe_col)[cols].transform("mean")
    yd = d[ycol] - means[ycol]
    Xd = d[xcols].values - means[xcols].values
    return cluster_ols(yd.values, Xd, d[cluster_col].values, add_const=False)


## 18.5 Act10 表计算

In [ ]:
wi_pre, wi_post = 15.23, 16.72
mn_pre, mn_post = 16.42, 18.10
did = (wi_post - wi_pre) - (mn_post - mn_pre)
print("DiD / beta =", did)
print("gamma (WI-MN pre) =", wi_pre - mn_pre)


## 18.6 CK1994 meal prices

In [ ]:
ck = pd.read_excel(ROOT / "CK1994/CK1994.xlsx")
for c in ck.columns:
    if c != "store":
        ck[c] = pd.to_numeric(ck[c], errors="coerce")
ck["price"] = ck["priceentree"] + ck["pricefry"] + ck["pricesoda"]
ck = ck.dropna(subset=["price"])
nper = ck.groupby("store").size()
ck = ck[ck.store.isin(nper[nper == 2].index)].copy()
ck["D"] = (ck["state"] * ck["time"]).astype(float)
ck["time1"] = ck["time"].astype(float)

print("Means by state x time:")
print(ck.groupby(["state", "time"])["price"].mean().unstack())
did = (
    ck[(ck.state == 1) & (ck.time == 1)]["price"].mean()
    - ck[(ck.state == 1) & (ck.time == 0)]["price"].mean()
    - (
        ck[(ck.state == 0) & (ck.time == 1)]["price"].mean()
        - ck[(ck.state == 0) & (ck.time == 0)]["price"].mean()
    )
)
print("DiD", did)

y = ck["price"].values
X = np.column_stack([ck["state"], ck["time"], ck["D"]])
b, se, V = cluster_ols(y, X, ck["store"].values, add_const=True)
print("\n(18.2) style:")
for n, bi, si in zip(["const", "state", "time", "D"], b, se):
    print(f"  {n:8s} {bi:8.4f} ({si:.4f})")

b, se, V = fe_reg(ck, "price", ["D", "time1"], "store", "store")
print("\nRestaurant FE:")
print(f"  D {b[0]:.4f} ({se[0]:.4f})")

ck["cent_time"] = ck["centralj"] * ck["time"]
ck["north_time"] = ck["northj"] * ck["time"]
b, se, V = fe_reg(ck, "price", ["D", "time1", "cent_time", "north_time"], "store", "store")
R = np.zeros((2, 4)); R[0, 2] = 1; R[1, 3] = 1
w = float((R @ b) @ inv(R @ V @ R.T) @ (R @ b))
print(f"\nHomogeneous treatment Wald p={1 - stats.chi2.cdf(w, 2):.3f}")

ck["pa1_time"] = ck["pa1"] * ck["time"]
b, se, V = fe_reg(ck, "price", ["D", "time1", "pa1_time"], "store", "store")
print(f"Equal control t={b[2]/se[2]:.3f}")


## 18.7 DS2004 police deterrence spillovers

In [ ]:
ds = pd.read_excel(ROOT / "DS2004/DS2004.xlsx")
for c in ds.columns:
    if c not in ["barrio", "calle", "altura"]:
        ds[c] = pd.to_numeric(ds[c], errors="coerce")
ds2 = ds[ds.month != 7].copy()
ds2["post"] = (ds2.month >= 8).astype(int)

for label, mask in [
    ("oneblock", ds2.oneblock == 1),
    ("farther", (ds2.oneblock == 0) & (ds2.sameblock == 0)),
    ("sameblock", ds2.sameblock == 1),
]:
    sub = ds2[mask]
    pre = sub[sub.post == 0]["thefts"].mean()
    post = sub[sub.post == 1]["thefts"].mean()
    print(f"{label:10s} pre {pre:.4f} post {post:.4f} d {post-pre:.4f}")

ds2["treat_same"] = ds2["sameblock"] * ds2["post"]
ds2["treat_one"] = ds2["oneblock"] * ds2["post"]
months = sorted(ds2.month.unique())
for m in months[1:]:
    ds2[f"m{m}"] = (ds2.month == m).astype(float)
xcols = ["treat_same", "treat_one"] + [f"m{m}" for m in months[1:]]
means = ds2.groupby("block")[["thefts"] + xcols].transform("mean")
yd = ds2["thefts"] - means["thefts"]
Xd = ds2[xcols].values - means[xcols].values
b, se, V = cluster_ols(yd.values, Xd, ds2["block"].values)
print("\nFE estimates:")
print(f"  sameblock x post {b[0]:.4f} ({se[0]:.4f})")
print(f"  oneblock x post  {b[1]:.4f} ({se[1]:.4f})")


## 18.8 BMN2016 beer / wine blue laws

In [ ]:
bmn = pd.read_stata(ROOT / "BMN2016/BMN2016.dta")


def panel_twoway(df, y, xcols, idcol, tcol):
    d = df.copy()
    times = sorted(d[tcol].unique())
    for t in times[1:]:
        d[f"t{int(t)}"] = (d[tcol] == t).astype(float)
    tdums = [f"t{int(t)}" for t in times[1:]]
    allx = xcols + tdums
    means = d.groupby(idcol)[[y] + allx].transform("mean")
    yd = d[y] - means[y]
    Xd = d[allx].values - means[allx].values
    b, se, V = cluster_ols(yd.values, Xd, d[idcol].values)
    return list(zip(allx, b, se))[: len(xcols)]


def panel_unit_trend(df, y, xcols, idcol, tcol):
    d = df.sort_values([idcol, tcol]).copy()
    d["_t"] = d[tcol].astype(float)
    vars_ = [y] + xcols
    out = {c: [] for c in vars_}
    ids, ts = [], []
    for i, g in d.groupby(idcol):
        t = g["_t"].values
        Tmat = np.column_stack([np.ones(len(t)), t])
        P = pinv(Tmat.T @ Tmat) @ Tmat.T
        for c in vars_:
            yc = g[c].values.astype(float)
            out[c].append(yc - Tmat @ (P @ yc))
        ids.append(np.full(len(t), i))
        ts.append(t)
    dd = pd.DataFrame({c: np.concatenate(out[c]) for c in vars_})
    dd[idcol] = np.concatenate(ids)
    dd[tcol] = np.concatenate(ts)
    return panel_twoway(dd, y, xcols, idcol, tcol)


df = bmn.dropna(subset=["logbeer", "beeronsun", "beeroffsun", "unempw", "beerOnOutflows", "beerOffOutflows"])
xcols = ["beeronsun", "beeroffsun", "unempw", "beerOnOutflows", "beerOffOutflows"]
print("Beer TWFE:")
for n, bi, si in panel_twoway(df, "logbeer", xcols, "id", "year"):
    print(f"  {n:16s} {bi:8.4f} ({si:.4f})")
print("Beer + unit trends:")
for n, bi, si in panel_unit_trend(df, "logbeer", xcols, "id", "year"):
    print(f"  {n:16s} {bi:8.4f} ({si:.4f})")

dfw = bmn.dropna(subset=["logwine", "wineonsun", "wineoffsun", "unempw", "wineOnOutflows", "wineOffOutflows"])
xcolsw = ["wineonsun", "wineoffsun", "unempw", "wineOnOutflows", "wineOffOutflows"]
print("\nWine TWFE:")
for n, bi, si in panel_twoway(dfw, "logwine", xcolsw, "id", "year"):
    print(f"  {n:16s} {bi:8.4f} ({si:.4f})")
print("Wine + unit trends:")
for n, bi, si in panel_unit_trend(dfw, "logwine", xcolsw, "id", "year"):
    print(f"  {n:16s} {bi:8.4f} ({si:.4f})")
